##### What is the Agent SDK?
###### Key Concepts
**Agents**: Autonomous programs that use Claude to accomplish tasks. They can reason, plan, and execute multi-step workflows without constant human input
**Tools**: Functions your agent can call -- both built-in (Read, Write, Bash) and custom tools you define via MCP or the @tool decorator.
**Orchestration**: Managing multi-step workflows where the agent plans its approach, executes steps in sequence, and adapts based on results.
**Guardrails**: Safety constraints and permission models that control what an agent can and cannot do, protecting against unintended side effects.

# Install the SDK 
pip install claude-agent-sdk 
# Verify installation
python3 -c "from claude_agent_sdk import query; print('SDK ready')"

In [ ]:
# query() -- Single-Exchange Interactions
from claude_agent_sdk import query

result = query(
    prompt="What files are in this directory?",
    options={
        "system_prompt": "You are a helpful file explorer.",
        "permission_mode": "default",
        "allowed_tools": ["Read", "Glob", "Grep"],
        "cwd": "/path/to/project"
    }
)
print(result.text)

In [ ]:
# Practical Example: Automated Code Review
from claude_agent_sdk import query

def review_pull_request(diff_text):
    """Automated code review for a pull request."""
    result = query(
        prompt=f"Review this code diff for bugs, security issues, and style:\n\n{diff_text}",
        options={
            "system_prompt": """You are a senior code reviewer. Focus on:
1. Security vulnerabilities
2. Logic errors
3. Performance issues
4. Code style consistency
Provide specific, actionable feedback.""",
            "allowed_tools": ["Read", "Grep"],
            "max_turns": 10
        }
    )
    return result.text

# Pro tip: Use max_turns to prevent the agent from spending too long on a task. For simple analysis, 5-10 turns is usually sufficient. For complex multi-file tasks, you may need 20-30.

In [ ]:
# 3. ClaudeSDKClient -- Multi-Turn Conversations
# For ongoing conversations where you need to maintain state between messages, ClaudeSDKClient provides a stateful session. Unlike query(), the client remembers previous messages and maintains context across multiple exchanges.
from claude_agent_sdk import ClaudeSDKClient

client = ClaudeSDKClient(options={
    "system_prompt": "You are a code review assistant.",
    "permission_mode": "acceptEdits"
})

# First message
response = client.send_message("Review the auth module for security issues.")
print(response.text)

# Follow-up (maintains conversation context)
response = client.send_message("Now check the same module for performance issues.")
print(response.text)

# Clean up
client.close()

In [ ]:
# Session Management
# Sessions can be paused and resumed using a session_id. This is useful for long-running workflows where you need to come back to a task later, or for building interactive applications where the user steps away and returns.
# Important: Always call client.close() when you are done with a session. The SDK spawns a subprocess that will continue running if not explicitly cleaned up. Use a try/finally block or a context manager pattern for safety.
# Create a named session
client = ClaudeSDKClient(options={
    "system_prompt": "You are a refactoring assistant.",
    "session_id": "refactor-auth-module-v2"
})

# Do some work...
response = client.send_message("Identify all functions that need refactoring in auth.py")

# Later, resume the same session
client2 = ClaudeSDKClient(options={
    "session_id": "refactor-auth-module-v2",
    "resume": True
})
response = client2.send_message("Now refactor the first function you identified.")

In [ ]:
# Agent SDK Playground

# Basic Query
from claude_agent_sdk import query

result = query(
    prompt="List all Python files in this project and\
            summarize what each one does.class="str">",
    options={
        "system_promptclass="str">": "You are a helpful code analyst.class="str">",
        "allowed_toolsclass="str">": ["Readclass="str">", "Globclass="str">", "Grepclass="str">"],
        "cwdclass="str">": "/home/user/my-projectclass="str">",
        "max_turns": 10
    }
)
print(result.text)

# Streaming
from claude_agent_sdk import query

class=class="str">"cm"># Stream output in real-time as the agent works
for event in query(
    prompt=class="str">"Refactor utils.py to use dataclasses",
    options={
        class="str">"system_prompt": class="str">"You are a Python refactoring expert.",
        class="str">"permission_mode": class="str">"acceptEdits",
        class="str">"allowed_tools": [class="str">"Read", class="str">"Edit", class="str">"Glob"],
        class="str">"cwd": class="str">"/home/user/my-project"
    },
    stream=True
):
    if event.type == class="str">"text":
        print(event.text, end=class="str">"")
    elif event.type == class="str">"tool_use":
        print(fclass="str">"\n[Using {event.tool}...]")
    elif event.type == class="str">"tool_result":
        print(fclass="str">"[{event.tool} complete]")

# Custom Tool
from claude_agent_sdk import query, tool
import json

@tool
def search_jira(
    project_key: str,
    status: str = class="str">"open"
) -> str:
    class=class="str">"str">class="str">""class="str">"Search JIRA issues in a project.

    Args:
        project_key: The JIRA project key (e.g., "PROJclass="str">")
        status: Filter by status: open, in_progress, done

    Returns:
    JSON string of matching issues
    "class="str">""
    class=class="str">"cm"># Simulated JIRA API call
    return json.dumps([
        {class="str">"id": class="str">"PROJ-101", class="str">"title": class="str">"Fix login bug",
         class="str">"priority": class="str">"high"},
        {class="str">"id": class="str">"PROJ-102", class="str">"title": class="str">"Add dark mode",
         class="str">"priority": class="str">"medium"}
    ])

result = query(
    prompt=class="str">"What are the open JIRA issues for PROJ?",
    options={
        class="str">"tools": [search_jira],
        class="str">"system_prompt": class="str">"Use the search_jira tool."
    }
)
print(result.text)

# ClaudeSDKClient
from claude_agent_sdk import ClaudeSDKClient

client = ClaudeSDKClient(options={
    class="str">"system_prompt": "You are a code review assistant.\
     Remember context from previous messages.class="str">",
    "permission_modeclass="str">": "acceptEditsclass="str">",
    "cwdclass="str">": "/home/user/my-projectclass="str">"
})

class="cmclass="str">"># First message
r1 = client.send_message(
    "Review auth.py for security issues.class="str">"
)
print("=== Review 1 ===class="str">")
print(r1.text)

class="cmclass="str">"># Follow-up with context
r2 = client.send_message(
    "Now check if those issues exist in api.py too.class="str">"
)
print("\n=== Review 2 ===")
print(r2.text)

client.close()

# HTL Aproval
from claude_agent_sdk import query

class = class="str">"cm"># Agent will pause for approval before
class = class="str">"cm"># destructive operations
result = query(
    prompt= class = "str">"Deploy the latest build to staging",
    options={
        class="str">"permission_mode": class="str">"default",
        class="str">"system_prompt": class=class="str">"str">class="str">""class="str">"
Before any deployment action:
1. Present your deployment plan
2. List all files to be deployed
3. Wait for explicit approval
4. Include a rollback plan

Never deploy without user confirmation.
"class="str">"",
        class="str">"allowed_tools": [
            class="str">"Read", class="str">"Glob", class="str">"Bash"
        ],
        class="str">"cwd": class="str">"/home/user/my-project"
    }
)
print(result.text)

# Permission Modes: 
class=class="str">"str">class="str">""class="str">"Compare permission modes side by side."class="str">""
from claude_agent_sdk import query

prompt = class="str">"Fix the bug in utils.py line 42"

class=class="str">"cm"># Mode 1: default (most conservative)
r1 = query(
    prompt=prompt,
    options={class="str">"permission_mode": class="str">"default"}
)
class=class="str">"cm"># Agent asks before: Read, Edit, Bash

class=class="str">"cm"># Mode 2: acceptEdits (balanced)
r2 = query(
    prompt=prompt,
    options={class="str">"permission_mode": class="str">"acceptEdits"}
)
class=class="str">"cm"># Agent auto-approves: Read, Edit
class=class="str">"cm"># Agent asks before: Bash

class=class="str">"cm"># Mode 3: fullAuto (most autonomous)
r3 = query(
    prompt=prompt,
    options={
        class="str">"permission_mode": class="str">"fullAuto",
        class="str">"allowed_tools": [
            class="str">"Read", class="str">"Edit", class="str">"Grep"
        ]  class=class="str">"cm"># ALWAYS restrict tools with fullAuto!
    }
)
class=class="str">"cm"># Agent auto-approves everything
class=class="str">"cm"># But can only use Read, Edit, Grep
Click

##### The Anatomy of a Great @tool Function
1. Descriptive function name -- the name itself tells Claude when to use it
2. Type-annotated parameters -- Python type hints become JSON Schema types
3. Default values -- optional parameters get default values
4. Rich docstring -- first line is the tool description, Args section describes each parameter
5. Return type hint -- tells Claude what to expect back
6. Structured return values -- JSON strings or plain text that Claude can parse

##### Tool Decorators
1. readOnly=True: Tool does not modify any state -- safe to auto-approve (e.g. Database queries, API reads, search operations)
2. destructive=True: Tool makes irreversible changes -- always ask for approval	(e.g. Deleting records, dropping tables, revoking access)
3. openWorld=True: Tool accesses external systems outside the local environment	(e.g. API calls, sending emails, posting to Slack)



In [ ]:
# 4. Custom Tools with @tool
# The @tool decorator is how you extend your agent's capabilities beyond Claude Code's built-in tools. Any Python function decorated with @tool becomes an MCP tool that the agent can discover and call during its reasoning process.

from claude_agent_sdk import tool
import json

@tool
def search_jira(project_key: str, status: str = "open") -> str:
    """Search JIRA issues in a project.

    Args:
        project_key: The JIRA project key (e.g., "PROJ")
        status: Filter by status. Options: open, in_progress, done, all

    Returns:
        JSON string of matching issues with id, title, assignee, and priority
    """
    issues = jira_client.search(project_key, status)
    return json.dumps([{
        "id": i.key,
        "title": i.summary,
        "assignee": i.assignee,
        "priority": i.priority
    } for i in issues])

In [ ]:
# Tool Annotations

@tool(annotations={"readOnly": True})
def get_user_profile(user_id: str) -> str:
    """Fetch a user's profile from the database.
    Args:
        user_id: The unique user identifier
    Returns:
        JSON string with user profile data
    """
    user = db.get_user(user_id)
    return json.dumps(user.to_dict())


@tool(annotations={"destructive": True})
def delete_user_account(user_id: str, reason: str) -> str:
    """Permanently delete a user account. This action is irreversible.

    Args:
        user_id: The unique user identifier
        reason: Reason for account deletion (for audit log)

    Returns:
        Confirmation message with deletion timestamp
    """
    result = db.delete_user(user_id, reason)
    return f"User {user_id} deleted at {result.timestamp}"

##### 5. Human-in-the-Loop (HITL) Patterns
###### Interrupt patterns
Approval Gates: The agent pauses before executing destructive or high-impact actions. It presents what it plans to do and waits for explicit human approval before proceeding.
Confirmation Prompts: "Are you sure you want to delete 500 records?" -- The agent detects potentially dangerous operations and surfaces a confirmation dialog before continuing.
Review Checkpoints: The agent presents a complete plan (e.g., "I will modify 12 files in the following way...") and waits for human review of the plan before executing any changes.

###### Implementation with Permission Modes
"default": Asks permission, Maximum safety, production environments
"acceptEdits": Auto-approved, Development workflows, code generation
"fullAuto": Auto-approved, CI/CD pipelines with strict tool restrictions

* Critical: Never use "fullAuto" without also restricting allowed_tools. An unrestricted agent in fullAuto mode can execute any shell command without asking -- this is powerful but dangerous if not properly scoped.

In [ ]:
# Implementation with Permission Modes

from claude_agent_sdk import query

# The system prompt instructs the agent to pause before destructive actions
result = query(
    prompt="Deploy the latest build to staging",
    options={
        "permission_mode": "default",  # Will pause for confirmations
        "system_prompt": """Before any deployment action, present your plan
and wait for approval. Include:
1. What will be deployed
2. Which environment
3. Expected impact
4. Rollback plan"""
    }
)

##### 6. Safety and Guardrails
##### Safety layers
Permission Models. Choose the right permission mode for your use case. Start with **"default"** and only relax restrictions as you gain confidence in the agent's behavior.

Tool Restrictions: Use **allowed_tools** to create a whitelist of permitted tools. This is your primary defense against unintended side effects.

Sandboxing: Run agents in containers, virtual machines, or other isolated environments. If an agent goes rogue, the blast radius is contained to the sandbox.

Monitoring and Logging: Log every tool call, every decision, and every output. Use this audit trail for debugging, compliance, and improving agent behavior over time.

Rate Limiting: Prevent runaway behavior by limiting the number of tool calls, the execution time, or the number of files modified. The **max_turns** option is your first line of defense here.

In [ ]:
# 6. Safety and Guardrails

# Read-only analysis agent -- cannot modify anything
result = query(
    prompt="Analyze this codebase for security vulnerabilities",
    options={
        "allowed_tools": ["Read", "Glob", "Grep"],  # No Write, Edit, or Bash
        "permission_mode": "default"
    }
)

In [ ]:
# The "Trust but Verify" Pattern
# Example: Restrict agent + monitor output
from claude_agent_sdk import query
import logging

logger = logging.getLogger("agent-audit")

result = query(
    prompt="Refactor the database module to use connection pooling",
    options={
        "permission_mode": "acceptEdits",
        "allowed_tools": ["Read", "Write", "Edit", "Glob", "Grep"],
        "max_turns": 25,
        "cwd": "/app/src"
    }
)

# Log the result for audit
logger.info("Agent task completed", extra={
    "result_length": len(result.text),
    "tool_calls": result.tool_call_count,
    "files_modified": result.files_modified
})